# Brain Tumor Classification
## EfficientNetV2-L Fine-Tuning

Author: Abdul Hafeez  
Model: EfficientNetV2-L  
Dataset: Brain_Tumor/Data_Split (70/15/15)  
Input Size: 224 × 224  
Classes: glioma, meningioma, pituitary, no_tumor  

Goal:
- Evaluate whether EfficientNetV2-L improves over EfficientNetV2-S
- Maintain generalization
- Avoid overfitting


In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# Enable mixed precision (RTX 3050 Ti safety)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow:", tf.__version__)
print("Mixed Precision Enabled")


2026-02-02 17:04:00.697435: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 17:04:04.687417: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-02 17:04:03.230040: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.20.0
Mixed Precision Enabled


In [2]:
BASE_DIR = "/mnt/c/Users/AbdulHafeez/Brain_Tumor/Data_Split"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 8      # if OOM → change to 4
NUM_CLASSES = 4

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]


In [3]:
print("Train exists:", os.path.exists(TRAIN_DIR))
print("Val exists:", os.path.exists(VAL_DIR))
print("Test exists:", os.path.exists(TEST_DIR))

for cls in CLASS_NAMES:
    print(cls, "→", len(os.listdir(os.path.join(TRAIN_DIR, cls))))


Train exists: True
Val exists: True
Test exists: True
glioma → 2628
meningioma → 1640
pituitary → 1894
no_tumor → 1230


In [4]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES
)

val_data = val_test_gen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
    classes=CLASS_NAMES
)

test_data = val_test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
    classes=CLASS_NAMES
)


Found 7392 images belonging to 4 classes.
Found 1584 images belonging to 4 classes.
Found 1584 images belonging to 4 classes.


In [5]:
base_model = tf.keras.applications.EfficientNetV2L(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False


I0000 00:00:1770051850.444204     467 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1753 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    dtype="float32"  # IMPORTANT for mixed precision
)(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ rescaling[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │      9,216 │ stem_activation[… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │        128 │ block1a_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 112, 112,  │          0 │ block1a_project_… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 112, 112,  │          0 │ block1a_project_… │
│                     │ 32)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 112, 112,  │      9,216 │ block1a_add[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 112, 112,  │        128 │ block1b_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 112, 112,  │          0 │ block1b_drop[0][… │
│                     │ 32)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_co… │ (None, 112, 112,  │      9,216 │ block1b_add[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_bn  │ (None, 112, 112,  │        128 │ block1c_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_ac… │ (None, 112, 112,  │          0 │ block1c_project_

 Total params: 117,757,092 (449.21 MB)

 Trainable params: 7,684 (30.02 KB)

 Non-trainable params: 117,749,408 (449.18 MB)

In [8]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [9]:
callbacks_stage1 = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]

history_stage1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=12,
    callbacks=callbacks_stage1
)


Epoch 1/12


/home/abdulhafeez/miniconda3/envs/tf/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
2026-02-02 17:04:50.477163: I external/local_xla/xla/service/service.cc:163] XLA service 0x7378fc015160 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-02 17:04:50.477191: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Ti Laptop GPU, Compute Capability 8.6
2026-02-02 17:04:51.525818: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-02 17:04:55.824984: I external/local_xla/xla/strea

924/924 ━━━━━━━━━━━━━━━━━━━━ 205s 128ms/step - accuracy: 0.4039 - loss: 1.3985 - val_accuracy: 0.5972 - val_loss: 0.9341
Epoch 2/12
924/924 ━━━━━━━━━━━━━━━━━━━━ 86s 93ms/step - accuracy: 0.4587 - loss: 1.2439 - val_accuracy: 0.6439 - val_loss: 0.9006
Epoch 3/12
924/924 ━━━━━━━━━━━━━━━━━━━━ 88s 96ms/step - accuracy: 0.4732 - loss: 1.2064 - val_accuracy: 0.6351 - val_loss: 0.8611
Epoch 4/12
924/924 ━━━━━━━━━━━━━━━━━━━━ 89s 96ms/step - accuracy: 0.4783 - loss: 1.2018 - val_accuracy: 0.5669 - val_loss: 0.9556
Epoch 5/12
924/924 ━━━━━━━━━━━━━━━━━━━━ 87s 94ms/step - accuracy: 0.4842 - loss: 1.2051 - val_accuracy: 0.6452 - val_loss: 0.8502
Epoch 6/12
924/924 ━━━━━━━━━━━━━━━━━━━━ 88s 95ms/step - accuracy: 0.4847 - loss: 1.1851 - val_accuracy: 0.6521 - val_loss: 0.8725
Epoch 7/12
924/924 ━━━━━━━━━━━━━━━━━━━━ 92s 100ms/step - accuracy: 0.4905 - loss: 1.1875 - val_accuracy: 0.6439 - val_loss: 0.8804
Epoch 8/12
924/924 ━━━━━━━━━━━━━━━━━━━━ 136s 98ms/step - accuracy: 0.4942 - loss: 1.1752 - val_acc

In [10]:
from tensorflow.keras import layers

for layer in base_model.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True


In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [12]:
callbacks_stage2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]


In [13]:
history_stage2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks_stage2
)


Epoch 1/20
924/924 ━━━━━━━━━━━━━━━━━━━━ 209s 127ms/step - accuracy: 0.4985 - loss: 1.1301 - val_accuracy: 0.6465 - val_loss: 0.8479 - learning_rate: 1.0000e-05
Epoch 2/20
924/924 ━━━━━━━━━━━━━━━━━━━━ 86s 93ms/step - accuracy: 0.5278 - loss: 1.0947 - val_accuracy: 0.6686 - val_loss: 0.8032 - learning_rate: 1.0000e-05
Epoch 3/20
924/924 ━━━━━━━━━━━━━━━━━━━━ 88s 95ms/step - accuracy: 0.5270 - loss: 1.0749 - val_accuracy: 0.6540 - val_loss: 0.8119 - learning_rate: 1.0000e-05
Epoch 4/20
924/924 ━━━━━━━━━━━━━━━━━━━━ 93s 101ms/step - accuracy: 0.5456 - loss: 1.0393 - val_accuracy: 0.6919 - val_loss: 0.8003 - learning_rate: 1.0000e-05
Epoch 5/20
924/924 ━━━━━━━━━━━━━━━━━━━━ 91s 98ms/step - accuracy: 0.5314 - loss: 1.0543 - val_accuracy: 0.6622 - val_loss: 0.7867 - learning_rate: 1.0000e-05
Epoch 6/20
924/924 ━━━━━━━━━━━━━━━━━━━━ 97s 105ms/step - accuracy: 0.5450 - loss: 1.0342 - val_accuracy: 0.6496 - val_loss: 0.8077 - learning_rate: 1.0000e-05
Epoch 7/20
  4/924 ━━━━━━━━━━━━━━━━━━━━ 1:31 99m

KeyboardInterrupt: 